In [1]:
using Pkg
Pkg.activate(".")

  Activating project at `~/code/compositional`


In [2]:
using DataFrames, JLD2
using CSV, CodecZlib

include("./Data_Input.jl")
using .DataImport

# Cross-sectional microbiome data

In [ ]:
sep_data = DataImport.GetCrossSecData("./Data/crosssecdata.RData"; min_samples=1, min_counts=1, min_nreads=1);

for key in keys(sep_data)
    df = copy(sep_data[key])
    select!(df, Not([:project_id, :sample_id]))
    rename!(df, :otu_id => :species_id, :run_id => :sample_id)
    df.class .= "microbiome"
    df.environment .= key
    select!(df, :class, :environment, :species_id, :sample_id, :count, :nreads)
    sep_data[key] = df
end

# Join multiple dfs together (if they have the same columns)
all_df = vcat(values(sep_data)...)

@save "./Data/microbiome.jld2" all_df
# later
# @load "all_data.jld2" all_df

In [41]:
Pkg.add("CSV")

   Resolving package versions...
  No Changes to `~/code/compositional/Project.toml`
  No Changes to `~/code/compositional/Manifest.toml`
Precompiling project...
  10037.4 ms  ✓ CSV
  1 dependency successfully precompiled in 11 seconds. 569 already precompiled.


In [10]:



# Path to your file
fname = "./Data/sra.gene_sums.SRP108713.G026.gz"

# Open a decompression stream over the gz file
gz = GzipDecompressorStream(open(fname, "r"))

# Read into a DataFrame
df = CSV.File(
    gz;
    delim='\t',
    header=3,
    strict=false,    # allow rows with varying column counts
    threaded=false   # avoid multi-thread parsing warnings
) |> DataFrame

close(gz)

┌ Warning: `threaded` keyword argument is deprecated; to avoid multithreaded parsing, pass `ntasks=1`
└ @ CSV ~/.julia/packages/CSV/XLcqT/src/context.jl:360


In [11]:
first(df, 5)

Row,gene_id,SRR5651390
,String31,Int64
1,ENSG00000278704.1,0
2,ENSG00000277400.1,0
3,ENSG00000274847.1,0
4,ENSG00000277428.1,0
5,ENSG00000276256.1,0
